In [1]:
import pandas as pd
import numpy as np
import json
import os
import warnings
warnings.filterwarnings('ignore')

In [2]:
df = pd.read_csv('../data/nassau_candy_cleaned.csv')
df['Order Date'] = pd.to_datetime(df['Order Date'])

print(f"Cleaned data loaded: {df.shape[0]} rows, {df.shape[1]} columns")

Cleaned data loaded: 6013 rows, 19 columns


### 1. Row Level KPIs

In [3]:
# These are calculated per transaction row

df['Gross_Margin_%']        = (df['Gross Profit'] / df['Sales']) * 100
df['Profit_per_Unit']       = df['Gross Profit'] / df['Units']
df['Cost_per_Unit']         = df['Cost'] / df['Units']
df['Revenue_Contribution_%']= (df['Sales'] / df['Sales'].sum()) * 100
df['Profit_Contribution_%'] = (df['Gross Profit'] / df['Gross Profit'].sum()) * 100

print("Row-level KPIs added:")
print("Gross_Margin_%")
print("Profit_per_Unit")
print("Cost_per_Unit")
print("Revenue_Contribution_%")
print("Profit_Contribution_%")

Row-level KPIs added:
Gross_Margin_%
Profit_per_Unit
Cost_per_Unit
Revenue_Contribution_%
Profit_Contribution_%


### 2. Business Level KPIs 

In [4]:
total_revenue       = df['Sales'].sum()
total_cost          = df['Cost'].sum()
total_profit        = df['Gross Profit'].sum()
total_units         = df['Units'].sum()
total_orders        = df['Order ID'].nunique()
overall_margin      = (total_profit / total_revenue) * 100
avg_order_value     = total_revenue / total_orders
avg_profit_per_unit = total_profit / total_units

print("BUSINESS LEVEL KPIs")
print(f"Total Revenue         : ${total_revenue:,.2f}")
print(f"Total Cost            : ${total_cost:,.2f}")
print(f"Total Gross Profit    : ${total_profit:,.2f}")
print(f"Total Units Sold      : {total_units:,}")
print(f"Total Orders          : {total_orders:,}")
print(f"Overall Gross Margin  : {overall_margin:.2f}%")
print(f"Avg Order Value       : ${avg_order_value:.2f}")
print(f"Avg Profit per Unit   : ${avg_profit_per_unit:.2f}")

BUSINESS LEVEL KPIs
Total Revenue         : $83,827.43
Total Cost            : $28,536.06
Total Gross Profit    : $55,291.37
Total Units Sold      : 22,755
Total Orders          : 5,067
Overall Gross Margin  : 65.96%
Avg Order Value       : $16.54
Avg Profit per Unit   : $2.43


### 3. Product Level KPIs

In [5]:
product_kpis = df.groupby(['Product Name', 'Division', 'Factory']).agg(
    Total_Revenue   = ('Sales', 'sum'),
    Total_Profit    = ('Gross Profit', 'sum'),
    Total_Cost      = ('Cost', 'sum'),
    Total_Units     = ('Units', 'sum'),
    Total_Orders    = ('Order ID', 'nunique'),
    Avg_Margin      = ('Gross_Margin_%', 'mean'),
    Avg_Profit_per_Unit = ('Profit_per_Unit', 'mean')
).round(2).reset_index()

product_kpis['Revenue_Share_%'] = (product_kpis['Total_Revenue'] / total_revenue * 100).round(2)
product_kpis['Profit_Share_%']  = (product_kpis['Total_Profit']  / total_profit  * 100).round(2)

product_kpis = product_kpis.sort_values('Total_Profit', ascending=False).reset_index(drop=True)

print("PRODUCT LEVEL KPIs")
print(product_kpis[['Product Name', 'Total_Revenue', 'Total_Profit', 
                     'Avg_Margin', 'Profit_Share_%']].to_string(index=False))

PRODUCT LEVEL KPIs
                     Product Name  Total_Revenue  Total_Profit  Avg_Margin  Profit_Share_%
   Wonka Bar -Scrumdiddlyumptious       16344.00      11350.00       69.44           20.53
Wonka Bar - Triple Dazzle Caramel       16680.00      10897.60       65.33           19.71
Wonka Bar - Nutty Crunch Surprise       14452.09      10311.09       71.35           18.65
       Wonka Bar - Milk Chocolate       15499.25      10062.59       64.92           18.20
        Wonka Bar - Fudge Mallows       14598.00       9732.00       66.67           17.60
               Lickable Wallpaper        4960.00       2480.00       50.00            4.49
                        Wonka Gum         328.75        170.95       52.00            0.31
           Everlasting Gobstopper         130.00        104.00       80.00            0.19
                      Hair Toffee          76.50         59.50       77.78            0.11
                        Kazookles         617.50         47.50        7

### 4. Division Level KPIs

In [6]:
division_kpis = df.groupby('Division').agg(
    Total_Revenue   = ('Sales', 'sum'),
    Total_Profit    = ('Gross Profit', 'sum'),
    Total_Cost      = ('Cost', 'sum'),
    Total_Units     = ('Units', 'sum'),
    Num_Products    = ('Product Name', 'nunique'),
    Avg_Margin      = ('Gross_Margin_%', 'mean')
).round(2).reset_index()

division_kpis['Revenue_Share_%'] = (division_kpis['Total_Revenue'] / total_revenue * 100).round(2)
division_kpis['Profit_Share_%']  = (division_kpis['Total_Profit']  / total_profit  * 100).round(2)

division_kpis = division_kpis.sort_values('Total_Profit', ascending=False).reset_index(drop=True)

print("DIVISION LEVEL KPIs")
print(division_kpis.to_string(index=False))

DIVISION LEVEL KPIs
 Division  Total_Revenue  Total_Profit  Total_Cost  Total_Units  Num_Products  Avg_Margin  Revenue_Share_%  Profit_Share_%
Chocolate       77573.34      52353.28    25220.06        21953             5       67.49            92.54           94.69
    Other        5906.25       2698.45     3207.80          701             3       39.01             7.05            4.88
    Sugar         347.84        239.64      108.20          101             7       58.75             0.41            0.43


### 5. Factory Level KPIs

In [7]:
factory_kpis = df.groupby('Factory').agg(
    Total_Revenue   = ('Sales', 'sum'),
    Total_Profit    = ('Gross Profit', 'sum'),
    Total_Cost      = ('Cost', 'sum'),
    Total_Units     = ('Units', 'sum'),
    Num_Products    = ('Product Name', 'nunique'),
    Avg_Margin      = ('Gross_Margin_%', 'mean')
).round(2).reset_index()

factory_kpis['Revenue_Share_%'] = (factory_kpis['Total_Revenue'] / total_revenue * 100).round(2)
factory_kpis['Profit_Share_%']  = (factory_kpis['Total_Profit']  / total_profit  * 100).round(2)

factory_kpis = factory_kpis.sort_values('Avg_Margin', ascending=False).reset_index(drop=True)

print("FACTORY LEVEL KPIs")
print(factory_kpis.to_string(index=False))

FACTORY LEVEL KPIs
          Factory  Total_Revenue  Total_Profit  Total_Cost  Total_Units  Num_Products  Avg_Margin  Revenue_Share_%  Profit_Share_%
    Lot's O' Nuts       45394.09      31393.09    14001.00        12736             3       69.20            54.15           56.78
  Wicked Choccy's       32179.25      20960.19    11219.06         9217             2       65.12            38.39           37.91
      Sugar Shack         141.34         76.14       65.20           71             5       52.09             0.17            0.14
   Secret Factory        5418.75       2754.95     2663.80          524             3       51.71             6.46            4.98
The Other Factory         694.00        107.00      587.00          207             2       12.88             0.83            0.19


### 6. Region Level KPIs

In [8]:
region_kpis = df.groupby('Region').agg(
    Total_Revenue   = ('Sales', 'sum'),
    Total_Profit    = ('Gross Profit', 'sum'),
    Total_Units     = ('Units', 'sum'),
    Total_Orders    = ('Order ID', 'nunique'),
    Avg_Margin      = ('Gross_Margin_%', 'mean')
).round(2).reset_index()

region_kpis['Revenue_Share_%'] = (region_kpis['Total_Revenue'] / total_revenue * 100).round(2)
region_kpis['Profit_Share_%']  = (region_kpis['Total_Profit']  / total_profit  * 100).round(2)

region_kpis = region_kpis.sort_values('Total_Profit', ascending=False).reset_index(drop=True)

print("REGION LEVEL KPIs")
print(region_kpis.to_string(index=False))

REGION LEVEL KPIs
  Region  Total_Revenue  Total_Profit  Total_Units  Total_Orders  Avg_Margin  Revenue_Share_%  Profit_Share_%
 Pacific       27560.35      18122.34         7401          1620       66.49            32.88           32.78
Atlantic       24237.06      15901.34         6546          1472       66.36            28.91           28.76
Interior       19343.79      12867.99         5279          1191       67.11            23.08           23.27
    Gulf       12686.23       8399.70         3529           784       66.53            15.13           15.19


### 7. Monthly KPIs

In [9]:
df['Month']      = df['Order Date'].dt.month
df['Month_Name'] = df['Order Date'].dt.strftime('%b')
df['Quarter']    = df['Order Date'].dt.quarter

monthly_kpis = df.groupby(['Quarter', 'Month', 'Month_Name']).agg(
    Total_Revenue = ('Sales', 'sum'),
    Total_Profit  = ('Gross Profit', 'sum'),
    Total_Orders  = ('Order ID', 'nunique'),
    Total_Units   = ('Units', 'sum')
).round(2).reset_index()

monthly_kpis['Margin_%'] = (monthly_kpis['Total_Profit'] / monthly_kpis['Total_Revenue'] * 100).round(2)
monthly_kpis = monthly_kpis.sort_values('Month').reset_index(drop=True)

print("MONTHLY KPIs")
print(monthly_kpis[['Month_Name', 'Quarter', 'Total_Revenue', 
                     'Total_Profit', 'Margin_%', 'Total_Orders']].to_string(index=False))

MONTHLY KPIs
Month_Name  Quarter  Total_Revenue  Total_Profit  Margin_%  Total_Orders
       Jan        1        3489.10       2328.81     66.75           209
       Feb        1        2553.65       1677.55     65.69           166
       Mar        1        5423.24       3588.38     66.17           356
       Apr        2        4922.44       3266.10     66.35           328
       May        2        6555.68       4329.10     66.04           399
       Jun        2        6214.93       4123.89     66.35           382
       Jul        3        6057.67       3983.05     65.75           365
       Aug        3        5740.10       3795.40     66.12           332
       Sep        3       11457.60       7498.83     65.45           698
       Oct        4        7345.89       4833.65     65.80           435
       Nov        4       11728.86       7750.66     66.08           701
       Dec        4       12338.27       8115.95     65.78           696


### 8. Save All KPIs to JSON

In [10]:
kpi_report = {
    "business_kpis": {
        "total_revenue"        : round(total_revenue, 2),
        "total_cost"           : round(total_cost, 2),
        "total_profit"         : round(total_profit, 2),
        "total_units"          : int(total_units),
        "total_orders"         : int(total_orders),
        "overall_gross_margin" : round(overall_margin, 2),
        "avg_order_value"      : round(avg_order_value, 2),
        "avg_profit_per_unit"  : round(avg_profit_per_unit, 2)
    },
    "product_kpis"  : product_kpis.to_dict(orient='records'),
    "division_kpis" : division_kpis.to_dict(orient='records'),
    "factory_kpis"  : factory_kpis.to_dict(orient='records'),
    "region_kpis"   : region_kpis.to_dict(orient='records'),
    "monthly_kpis"  : monthly_kpis.to_dict(orient='records')
}

report_path = '../outputs/reports/kpi_report.json'
os.makedirs(os.path.dirname(report_path), exist_ok=True)

with open(report_path, 'w') as f:
    json.dump(kpi_report, f, indent=4)

print(f"KPI report saved to: {report_path}")

KPI report saved to: ../outputs/reports/kpi_report.json


### 9. Save Enriched Dataset

In [11]:
# Save df with all new KPI columns for use in analysis notebooks

enriched_path = '../data/nassau_candy_enriched.csv'
df.to_csv(enriched_path, index=False)

print(f"Enriched dataset saved to: {enriched_path}")
print(f"Columns added: Gross_Margin_%, Profit_per_Unit, Cost_per_Unit,")
print(f"Revenue_Contribution_%, Profit_Contribution_%,")
print(f"Month, Month_Name, Quarter")

Enriched dataset saved to: ../data/nassau_candy_enriched.csv
Columns added: Gross_Margin_%, Profit_per_Unit, Cost_per_Unit,
Revenue_Contribution_%, Profit_Contribution_%,
Month, Month_Name, Quarter
